In [1]:
import pandas as pd
import numpy as np
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
from IPython.display import display

In [2]:
spark = SparkSession.builder.master("local[*]") \
.appName('Spark_App') \
.getOrCreate()

In [3]:
import datetime
from pyspark.sql import Window
from pyspark.sql.functions import lit, col, when, log
from pyspark.ml.feature import StringIndexer, OneHotEncoder, MaxAbsScaler, VectorAssembler, SQLTransformer
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import matplotlib.pyplot as plt

## Step 1. Prepare the data

In [4]:
credit_data = pd.read_hdf('credit_data.h5', 'df')
credit_data_sdf = spark.createDataFrame(credit_data)

## Data Exploration

In [5]:
print(credit_data_sdf.count())

5000


In [6]:
display(credit_data_sdf.select('*').limit(5).toPandas())

,CustomerID,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,713a336c-a255-4e2d-9d57-90b3e99e2f06,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,140b363f-a3fe-4828-a33f-7284dfdb3969,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,43b7b51d-5eda-4860-b461-ebef3d3436f4,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,f40eaf08-e6d1-4765-ab20-c5f7faca1635,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,1728910a-d3ff-4799-ac50-203a3a58a3fb,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,...,unknown,57,none,own,2,skilled,1,none,yes,Risk


In [7]:
display(credit_data_sdf
        .select("LoanAmount")
        .summary("min", "10%", "20%", "30%", "40%", "50%", "60%", "70%", "80%", "90%", "max")
        .toPandas())

,summary,LoanAmount
0,min,250
1,10%,250
2,20%,858
3,30%,1749
4,40%,2473
5,50%,3237
6,60%,4059
7,70%,4905
8,80%,5816
9,90%,6898


In [8]:
display(credit_data_sdf
        .groupBy("LoanPurpose")
        .count()
        .sort("count", ascending=False)
        .toPandas())

,LoanPurpose,count
0,car_new,945
1,furniture,853
2,car_used,808
3,radio_tv,755
4,appliances,561
5,repairs,283
6,vacation,205
7,education,167
8,retraining,164
9,business,146


## Background: Transformers, estimators, and pipelines
## Step 2. Feature preprocessing
## Convert categorical variables to numeric

In [9]:
credit_data_sdf.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- CheckingStatus: string (nullable = true)
 |-- LoanDuration: long (nullable = true)
 |-- CreditHistory: string (nullable = true)
 |-- LoanPurpose: string (nullable = true)
 |-- LoanAmount: long (nullable = true)
 |-- ExistingSavings: string (nullable = true)
 |-- EmploymentDuration: string (nullable = true)
 |-- InstallmentPercent: long (nullable = true)
 |-- Sex: string (nullable = true)
 |-- OthersOnLoan: string (nullable = true)
 |-- CurrentResidenceDuration: long (nullable = true)
 |-- OwnsProperty: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- InstallmentPlans: string (nullable = true)
 |-- Housing: string (nullable = true)
 |-- ExistingCreditsCount: long (nullable = true)
 |-- Job: string (nullable = true)
 |-- Dependents: long (nullable = true)
 |-- Telephone: string (nullable = true)
 |-- ForeignWorker: string (nullable = true)
 |-- Risk: string (nullable = true)



In [10]:
#Get All column names and it's types
for field in credit_data_sdf.schema.fields:
    print(field.name +", "+str(field.dataType))

CustomerID, StringType()
CheckingStatus, StringType()
LoanDuration, LongType()
CreditHistory, StringType()
LoanPurpose, StringType()
LoanAmount, LongType()
ExistingSavings, StringType()
EmploymentDuration, StringType()
InstallmentPercent, LongType()
Sex, StringType()
OthersOnLoan, StringType()
CurrentResidenceDuration, LongType()
OwnsProperty, StringType()
Age, LongType()
InstallmentPlans, StringType()
Housing, StringType()
ExistingCreditsCount, LongType()
Job, StringType()
Dependents, LongType()
Telephone, StringType()
ForeignWorker, StringType()
Risk, StringType()


In [11]:
numericalCols = [
    'LoanDuration',
    'LoanAmount',
    'Age',
    'InstallmentPercent',
    'CurrentResidenceDuration'
]

categoricalCols = [
    'ExistingCreditsCount',
    'Dependents',
    'CheckingStatus',
    'CreditHistory',
    'LoanPurpose',
    'ExistingSavings',
    'EmploymentDuration',
    'Sex',
    'OthersOnLoan',
    'OwnsProperty',
    'InstallmentPlans',
    'Housing',
    'Job',
    'Telephone',
    'ForeignWorker'    
]

In [12]:
# Transformation which are defined by SQL statement
sqlTrans = SQLTransformer(statement = "SELECT * FROM __THIS__")

In [13]:
# StringIndexer Initialization
stringIndexer = StringIndexer(inputCols = categoricalCols, outputCols=[x + "Index" for x in categoricalCols], stringOrderType = 'alphabetAsc')

# OneHotEncoder Initialization
encoder = OneHotEncoder(inputCols = stringIndexer.getOutputCols(), outputCols=["onehot" + x for x in categoricalCols], dropLast = False)

In [14]:
# StringIndexer Initialization
labelToIndex = StringIndexer(inputCol="Risk", outputCol="label", stringOrderType = 'alphabetAsc')

In [15]:
#Fits a model to the input dataset with optional parameters.
df0 = sqlTrans.transform(credit_data_sdf)
df1 = stringIndexer.fit(df0).transform(df0)
display(df1.select('OwnsProperty', 'OwnsPropertyIndex').limit(10).toPandas())

,OwnsProperty,OwnsPropertyIndex
0,savings_insurance,2.0
1,savings_insurance,2.0
2,real_estate,1.0
3,savings_insurance,2.0
4,unknown,3.0
5,unknown,3.0
6,savings_insurance,2.0
7,car_other,0.0
8,savings_insurance,2.0
9,unknown,3.0


In [16]:
#Fits a model to the input dataset with optional parameters.
df2 = encoder.fit(df1).transform(df1)
display(df2.select('OwnsProperty', 'OwnsPropertyIndex', 'onehotOwnsProperty').limit(10).toPandas())

,OwnsProperty,OwnsPropertyIndex,onehotOwnsProperty
0,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)"
1,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)"
2,real_estate,1.0,"(0.0, 1.0, 0.0, 0.0)"
3,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)"
4,unknown,3.0,"(0.0, 0.0, 0.0, 1.0)"
5,unknown,3.0,"(0.0, 0.0, 0.0, 1.0)"
6,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)"
7,car_other,0.0,"(1.0, 0.0, 0.0, 0.0)"
8,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)"
9,unknown,3.0,"(0.0, 0.0, 0.0, 1.0)"


In [17]:
df3 = labelToIndex.fit(df0).transform(df0)
display(df3.select('Risk', 'label').limit(10).toPandas())

,Risk,label
0,No Risk,0.0
1,No Risk,0.0
2,No Risk,0.0
3,No Risk,0.0
4,Risk,1.0
5,Risk,1.0
6,No Risk,0.0
7,No Risk,0.0
8,No Risk,0.0
9,Risk,1.0


## Combine all feature columns into a single feature vector

In [18]:
# This includes both the numeric columns and the one-hot encoded binary vector columns in our dataset.
assemblerInputs =  numericalCols + ["onehot" + c for c in categoricalCols]

# VectorAssembler Initialization
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol = "features")

In [19]:
# Define the pipeline model.
pipeline = Pipeline(stages=[sqlTrans, stringIndexer, encoder, vecAssembler, labelToIndex])

In [20]:
df_pipeline = pipeline.fit(credit_data_sdf).transform(credit_data_sdf)
display(df_pipeline.select('LoanDuration', 'OwnsProperty', 'OwnsPropertyIndex', 'onehotOwnsProperty', 'features', 'label').limit(10).toPandas())

,LoanDuration,OwnsProperty,OwnsPropertyIndex,onehotOwnsProperty,features,label
0,31,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)","(31.0, 1889.0, 32.0, 3.0, 3.0, 1.0, 0.0, 0.0, ...",0.0
1,18,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)","(18.0, 462.0, 37.0, 2.0, 2.0, 0.0, 1.0, 0.0, 0...",0.0
2,15,real_estate,1.0,"(0.0, 1.0, 0.0, 0.0)","(15.0, 250.0, 28.0, 2.0, 3.0, 0.0, 1.0, 0.0, 0...",0.0
3,28,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)","(28.0, 3693.0, 32.0, 3.0, 2.0, 1.0, 0.0, 0.0, ...",0.0
4,28,unknown,3.0,"(0.0, 0.0, 0.0, 1.0)","(28.0, 6235.0, 57.0, 3.0, 3.0, 0.0, 1.0, 0.0, ...",1.0
5,32,unknown,3.0,"(0.0, 0.0, 0.0, 1.0)","(32.0, 9604.0, 57.0, 6.0, 5.0, 0.0, 1.0, 0.0, ...",1.0
6,9,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)","(9.0, 1032.0, 41.0, 3.0, 4.0, 1.0, 0.0, 0.0, 0...",0.0
7,16,car_other,0.0,"(1.0, 0.0, 0.0, 0.0)","(16.0, 3109.0, 36.0, 3.0, 1.0, 0.0, 1.0, 0.0, ...",0.0
8,11,savings_insurance,2.0,"(0.0, 0.0, 1.0, 0.0)","(11.0, 4553.0, 22.0, 3.0, 3.0, 1.0, 0.0, 0.0, ...",0.0
9,35,unknown,3.0,"(0.0, 0.0, 0.0, 1.0)","(35.0, 7138.0, 49.0, 5.0, 4.0, 0.0, 1.0, 0.0, ...",1.0


## Step 3. Create Decision Tree Model

In [21]:
# Decision Tree Classifier
params = {
    "maxDepth": 9,
    "maxBins": 32,
    "minInstancesPerNode": 10,
    "impurity": "gini",
    "seed": 42
}

dtc = DecisionTreeClassifier(**params)

## Step 4. Build the pipeline

In [22]:
# Define the pipeline based on the stages created in previous steps.
pipeline = Pipeline(stages=[sqlTrans, stringIndexer, encoder, vecAssembler, labelToIndex, dtc])
 
# Fit the model
dt_model = pipeline.fit(credit_data_sdf)
 
# Make predictions
df_predictions = dt_model.transform(credit_data_sdf)

In [23]:
display(df_predictions.select("features", "label", "Risk", "probability", "prediction").limit(10).toPandas())

,features,label,Risk,probability,prediction
0,"(31.0, 1889.0, 32.0, 3.0, 3.0, 1.0, 0.0, 0.0, ...",0.0,No Risk,"[0.9323308270676691, 0.06766917293233082]",0.0
1,"(18.0, 462.0, 37.0, 2.0, 2.0, 0.0, 1.0, 0.0, 0...",0.0,No Risk,"[0.8719512195121951, 0.12804878048780488]",0.0
2,"(15.0, 250.0, 28.0, 2.0, 3.0, 0.0, 1.0, 0.0, 0...",0.0,No Risk,"[0.9851190476190477, 0.01488095238095238]",0.0
3,"(28.0, 3693.0, 32.0, 3.0, 2.0, 1.0, 0.0, 0.0, ...",0.0,No Risk,"[0.8731343283582089, 0.12686567164179105]",0.0
4,"(28.0, 6235.0, 57.0, 3.0, 3.0, 0.0, 1.0, 0.0, ...",1.0,Risk,"[0.07894736842105263, 0.9210526315789473]",1.0
5,"(32.0, 9604.0, 57.0, 6.0, 5.0, 0.0, 1.0, 0.0, ...",1.0,Risk,"[0.04833836858006042, 0.9516616314199395]",1.0
6,"(9.0, 1032.0, 41.0, 3.0, 4.0, 1.0, 0.0, 0.0, 0...",0.0,No Risk,"[0.810077519379845, 0.18992248062015504]",0.0
7,"(16.0, 3109.0, 36.0, 3.0, 1.0, 0.0, 1.0, 0.0, ...",0.0,No Risk,"[0.8719512195121951, 0.12804878048780488]",0.0
8,"(11.0, 4553.0, 22.0, 3.0, 3.0, 1.0, 0.0, 0.0, ...",0.0,No Risk,"[0.9552238805970149, 0.04477611940298507]",0.0
9,"(35.0, 7138.0, 49.0, 5.0, 4.0, 0.0, 1.0, 0.0, ...",1.0,Risk,"[0.04833836858006042, 0.9516616314199395]",1.0


## Step 5. Visualize the Decision Tree

In [24]:
print(dt_model.stages[-1].toDebugString)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_cccc16adde20, depth=9, numNodes=191, numClasses=2, numFeatures=64
  If (feature 14 in {0.0})
   If (feature 2 <= 32.5)
    If (feature 0 <= 18.5)
     If (feature 47 in {1.0})
      Predict: 0.0
     Else (feature 47 not in {1.0})
      If (feature 4 <= 2.5)
       Predict: 0.0
      Else (feature 4 > 2.5)
       If (feature 39 in {1.0})
        Predict: 0.0
       Else (feature 39 not in {1.0})
        If (feature 11 in {0.0})
         Predict: 0.0
        Else (feature 11 not in {0.0})
         If (feature 34 in {0.0})
          Predict: 0.0
         Else (feature 34 not in {0.0})
          If (feature 37 in {1.0})
           Predict: 1.0
          Else (feature 37 not in {1.0})
           Predict: 0.0
    Else (feature 0 > 18.5)
     If (feature 39 in {1.0})
      Predict: 0.0
     Else (feature 39 not in {1.0})
      If (feature 12 in {1.0})
       If (feature 2 <= 27.5)
        Predict: 0.0
       Else (feature 2 > 27.5)


## Step 6. Feature Importance

In [25]:
feature_importance = dt_model.stages[-1].featureImportances.toArray()

# Show feature importance
for i, column in enumerate(dt_model.stages[-3].getInputCols()):
    print(f"Feature '{column}': {feature_importance[i]:.4f}")

Feature 'LoanDuration': 0.0655
Feature 'LoanAmount': 0.0186
Feature 'Age': 0.0993
Feature 'InstallmentPercent': 0.0040
Feature 'CurrentResidenceDuration': 0.0131
Feature 'onehotExistingCreditsCount': 0.0256
Feature 'onehotDependents': 0.0022
Feature 'onehotCheckingStatus': 0.0000
Feature 'onehotCreditHistory': 0.0000
Feature 'onehotLoanPurpose': 0.0000
Feature 'onehotExistingSavings': 0.0000
Feature 'onehotEmploymentDuration': 0.0088
Feature 'onehotSex': 0.0041
Feature 'onehotOthersOnLoan': 0.0097
Feature 'onehotOwnsProperty': 0.3618
Feature 'onehotInstallmentPlans': 0.0000
Feature 'onehotHousing': 0.0000
Feature 'onehotJob': 0.0000
Feature 'onehotTelephone': 0.0064
Feature 'onehotForeignWorker': 0.0000


## Step 7. Evaluate the model

In [26]:
bcEvaluator = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
auc = bcEvaluator.evaluate(df_predictions)
gini = 2.0*auc-1.0
print(f"Area under ROC curve: {auc:.6f}")
print(f"Gini: {gini:.6f}")
mcEvaluator = MulticlassClassificationEvaluator(labelCol='label', metricName='f1')
f1 = mcEvaluator.evaluate(df_predictions)
print(f"F1 Score: {f1:.6f}")

Area under ROC curve: 0.680059
Gini: 0.360119
F1 Score: 0.826628
